<div class="alert alert-info" markdown="1">

#### Homework 5, Problem 11 Supplemental Notebook

# Matrices and Linear Transformations

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="https://math124.org">math124.org</a></small>

</div>

### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

Open this notebook in Google Colab. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

Together, the tasks in this notebook constitute your score for Homework 5, Problem 11, worth **20 points**. You won't need to submit this notebook anywhere. Instead, at the bottom, we'll tell you what to include in your PDF submission for Homework 5 that you upload to Pensive.

Work from top to bottom. **We've done the first color filter (grayscale) and the first geometric transformation (stretching) for you.** Run those examples and use them as models for your own functions. Replace the `...` in the remaining function cells. All of the code for the buttons, sliders, and canvas is supplied.

If you change a function, run its cell again and rerun the checks and canvas cells below it. The buttons and sliders use the functions from the last time you ran that canvas cell.

Task 1 is required practice worth 0 points. Tasks 2–5 are worth 5 points each, for a total of 20 points.

## Canva124: your image editing app

---

Your goal here is to build a very basic image editing app, using what you've learned about linear transformations and matrix multiplication. We'll call it **Canva124**. Each task adds another feature to the same app.

Canva124 has two groups of buttons: **Colors** and **Transformations**. Select one color filter and one geometric transformation at a time. Click a transformation to reveal its parameter sliders; click **Tint** to choose a color.

Click **Upload image** in any Canva124 canvas to use your own picture. It automatically resizes the image to fit, keeping its proportions and limiting its longest side to 960 pixels. PNG, JPEG, GIF, and WebP files work. You can also keep the included sample image. Rerunning a canvas cell returns it to the sample; upload your picture again if needed.

We've done grayscale and stretching for you. Run these examples, then use them as models for your own functions. All of the buttons, sliders, color picker, and drawing code are supplied.

## Setup

---

Run the supplied cell below. It loads NumPy, a small sample image, and the Canva124 tools. You do not need to edit it.

**NumPy reminder:** `np.array([[1, 2], [3, 4]])` makes a matrix, and `A @ u` multiplies it by a vector. `*` multiplies matching entries instead. `np.eye(2)` is the $2\times2$ identity matrix. In our code, a one-dimensional array such as `np.array([2, 3])` represents the column vector $\begin{bmatrix}2\\3\end{bmatrix}$.

The supplied display code rounds color values and clips them to $[0,255]$ **only when drawing**. Your functions should return unrounded numerical results, including values outside that interval.

In [ ]:
#@title Load the sample image and Canva124 tools { display-mode: "form" }
"""Supplied display tools, embedded in the student notebook at build time."""
import base64
import io
import json
import uuid
import numpy as np
from PIL import Image, ImageOps
from IPython.display import HTML, display

def _matrix(value, shape, name):
    if value is None or value is Ellipsis:
        raise NotImplementedError(name)
    value = np.asarray(value, dtype=float)
    if value.shape != shape or not np.isfinite(value).all():
        raise ValueError(f'{name} must return a finite array with shape {shape}.')
    return value

def _png(pixels):
    # Clip and round only when displaying; students' mathematical outputs stay raw.
    pixels = np.rint(np.clip(pixels, 0, 255)).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(pixels).save(buffer, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode('ascii')

def check_task(task):
    """Small visible checks. These are examples, not a complete proof of correctness."""
    try:
        if task == 1:
            A = np.array([[1., 2., 0.], [0., 1., 0.], [0., 0., -1.]])
            u = np.array([10., 20., 30.])
            np.testing.assert_allclose(_matrix(filter_pixel(A, u), (3,), 'filter_pixel'), [50, 20, -30])
            G = _matrix(grayscale_matrix(), (3, 3), 'grayscale_matrix')
            for u in [np.array([50., 200., 14.]), np.array([90., 0., 0.]), np.array([0., 30., 0.])]:
                np.testing.assert_allclose(filter_pixel(G, u), np.repeat(np.mean(u), 3))
        elif task == 2:
            S = _matrix(sepia_matrix(), (3, 3), 'sepia_matrix')
            np.testing.assert_allclose(filter_pixel(S, np.array([50., 200., 14.])), [176.096, 157.002, 122.234])
            # This should exceed 255: clipping belongs to the supplied display code.
            np.testing.assert_allclose(filter_pixel(S, np.array([255., 255., 255.])), [344.505, 306.765, 238.935])
        elif task == 3:
            for rgb in [(255, 255, 255), (0, 0, 0), (255, 153, 51), (0, 255, 0)]:
                A = _matrix(tint_matrix(rgb), (3, 3), 'tint_matrix')
                np.testing.assert_allclose(A, np.diag(np.asarray(rgb) / 255))
            np.testing.assert_allclose(filter_pixel(tint_matrix((255, 153, 51)), np.array([80., 120., 200.])), [80, 72, 40])
        elif task == 4:
            X = np.array([[0., 1., 0., 1.], [0., 0., 1., 1.]])
            A = np.array([[2., -1.], [1., 3.]])
            np.testing.assert_allclose(_matrix(transform_points(A, X), (2, 4), 'transform_points'), A @ X)
            for k in [-1., 0., .5]:
                H = _matrix(shear_matrix(k), (2, 2), 'shear_matrix')
                np.testing.assert_allclose(H @ np.array([2., 3.]), [2 + 3*k, 3])
        elif task == 5:
            Q = _matrix(rotation_matrix(np.pi/2), (2, 2), 'rotation_matrix')
            np.testing.assert_allclose(Q @ np.array([2., 1.]), [-1, 2], atol=1e-12)
            np.testing.assert_allclose(rotation_matrix(-np.pi/2) @ np.array([2., 1.]), [1, -2], atol=1e-12)
            np.testing.assert_allclose(rotation_matrix(0), np.eye(2), atol=1e-12)
            D = _matrix(scale_matrix(2., .5), (2, 2), 'scale_matrix')
            np.testing.assert_allclose(D @ np.array([2., 4.]), [4, 2])
        else:
            raise ValueError('Task must be 1 through 5.')
    except (NotImplementedError, NameError) as error:
        print(f'Task {task}: complete and run the requested function cells first. {error}')
        return False
    except (AssertionError, ValueError, TypeError, IndexError) as error:
        print(f'Task {task}: a check needs attention.\n{error}')
        return False
    print(f'Task {task}: all example checks passed. Now explore the canvas.')
    return True

def _color_operator(A):
    # Export the linear map defined by the supplied pixel function. This lets
    # the browser apply the same map to newly uploaded pixels without a kernel call.
    basis = np.eye(3)
    return np.column_stack([_matrix(filter_pixel(A.copy(), u.copy()), (3,), 'filter_pixel')
                            for u in basis.T])

def _studio_html(stage, tools=None):
    colors = [{'label': 'Original', 'matrix': np.eye(3).tolist(), 'operator': np.eye(3).tolist()}]
    for enabled, label, name in [(stage >= 1, 'Grayscale', 'grayscale_matrix'),
                                  (stage >= 2, 'Sepia', 'sepia_matrix')]:
        if enabled:
            A = _matrix(globals()[name](), (3, 3), name)
            colors.append({'label': label, 'matrix': A.tolist(), 'operator': _color_operator(A).tolist()})
    tint_basis = []
    if stage >= 3:
        # The specified tint matrix depends linearly on the three selected RGB
        # values. Cache each channel's contribution from the student's function.
        for rgb in [(255, 0, 0), (0, 255, 0), (0, 0, 255)]:
            A = _matrix(tint_matrix(rgb), (3, 3), 'tint_matrix')
            tint_basis.append({'matrix': A.tolist(), 'operator': _color_operator(A).tolist()})
        for rgb in [(0, 0, 0), (255, 255, 255), (32, 149, 217)]:
            expected = sum((value / 255) * np.asarray(item['matrix']) for value, item in zip(rgb, tint_basis))
            if not np.allclose(tint_matrix(rgb), expected):
                raise ValueError('Each tint-matrix entry must depend linearly on the selected RGB values. Check tint_matrix.')
    operations = []
    def add(kind, parameters, label, A):
        A = _matrix(A, (2, 2), kind)
        operator = _matrix(transform_points(A.copy(), np.eye(2)), (2, 2), 'transform_points') if stage >= 4 else A
        operations.append({'kind': kind, 'parameters': parameters, 'label': label,
                           'matrix': A.tolist(), 'operator': operator.tolist()})
    add('original', {}, 'Original position', np.eye(2))
    enabled = set(tools) if tools is not None else {'stretch', 'shear', 'rotate'}
    if stage >= 4 and 'stretch' in enabled:
        for sx in np.arange(.5, 2.51, .25):
            for sy in np.arange(.5, 2.51, .25):
                add('stretch', {'sx': float(sx), 'sy': float(sy)}, f'Stretch: x × {sx:g}, y × {sy:g}', scale_matrix(sx, sy))
    if stage >= 4 and 'shear' in enabled:
        for k in np.arange(-2., 2.01, .25):
            add('shear', {'k': float(k)}, f'Shear: k = {k:g}', shear_matrix(k))
    if stage >= 5 and 'rotate' in enabled:
        for angle in range(-180, 181, 15):
            add('rotate', {'angle': angle}, f'Rotate: {angle}°', rotation_matrix(np.deg2rad(angle)))
    payload = {'stage': stage, 'sample': _png(np.asarray(studio_image, dtype=float)),
               'colors': colors, 'tint': tint_basis, 'ops': operations}
    return _CANVAS_TEMPLATE.replace('__ID__', 'canva124_' + uuid.uuid4().hex).replace('__DATA__', json.dumps(payload))

def show_studio(stage, tools=None):
    """Display Canva124 using the functions completed through this task."""
    if not 1 <= stage <= 5:
        raise ValueError('Task must be 1 through 5.')
    try:
        display(HTML(_studio_html(stage, tools)))
    except (NotImplementedError, NameError, ValueError, TypeError, IndexError) as error:
        print(f'Complete the functions through Task {stage}, run their cells, then rerun Canva124.\n{error}')

_CANVAS_TEMPLATE = r'''<div id="__ID__" class="canva124">
<style>
#__ID__ {max-width:1050px;margin:14px auto;padding:20px;border:1px solid #cbd5e1;border-radius:12px;background:#fff;color:#15253a;font:16px/1.45 Georgia,serif;box-sizing:border-box}
#__ID__ * {box-sizing:border-box}
#__ID__ h3 {margin:0;font-size:26px} #__ID__ .hint {color:#536278;font-size:14px;margin:4px 0 12px}
#__ID__ button, #__ID__ .upload {font:14px system-ui,sans-serif;border:1px solid #aab8c8;border-radius:5px;padding:8px 12px;background:white;color:#15253a;min-height:38px;cursor:pointer}
#__ID__ button:hover, #__ID__ .upload:hover {background:#edf4fc}
#__ID__ button[aria-pressed="true"] {background:#00274c;color:#ffcb05;border-color:#00274c}
#__ID__ .row {display:flex;gap:8px;flex-wrap:wrap;align-items:center;margin:8px 0}
#__ID__ .upload {display:inline-flex;align-items:center} #__ID__ .file-input {width:1px;height:1px;position:absolute;opacity:0}
#__ID__ .file-input:focus + span {outline:2px solid #306b92}
#__ID__ .tool-group {margin:10px 0;padding:11px 13px;background:#f7f9fc;border:1px solid #e2e8f0;border-radius:8px}
#__ID__ .sliders label {display:grid;grid-template-columns:155px 1fr 48px;gap:10px;align-items:center;margin-top:10px;font:14px system-ui,sans-serif}
#__ID__ input[type="range"] {width:100%;accent-color:#00274c;cursor:pointer}
#__ID__ input[type="color"] {width:55px;height:36px;cursor:pointer;border:1px solid #aab8c8;padding:2px;border-radius:4px;background:white}
#__ID__ .tint-controls {display:flex;align-items:center;gap:12px;flex-wrap:wrap;font:14px system-ui,sans-serif;margin-top:10px}
#__ID__ canvas {width:100%;height:auto;display:block;border:1px solid #e2e8f0;border-radius:6px}
#__ID__ .matrices {display:flex;flex-wrap:wrap;gap:24px;justify-content:center;padding-top:12px;font-size:14px}
#__ID__ .matrix {display:inline-grid;position:relative;gap:3px 14px;padding:5px 10px;text-align:right;vertical-align:middle;font:14px/1.5 ui-monospace,monospace;margin-left:8px}
#__ID__ .matrix:before,#__ID__ .matrix:after {content:'';position:absolute;top:0;bottom:0;width:5px;border-top:1px solid;border-bottom:1px solid}
#__ID__ .matrix:before {left:0;border-left:1px solid} #__ID__ .matrix:after {right:0;border-right:1px solid}
#__ID__ .status {font:13px system-ui,sans-serif;color:#536278;margin:7px 0}
#__ID__ [hidden] {display:none!important}
</style>
<h3>Canva124</h3>
<p class="hint">After editing a function, rerun this canvas cell.</p>
<div class="row"><label class="upload"><input class="file-input" type="file" accept="image/png,image/jpeg,image/webp,image/gif" aria-label="Upload image"><span>Upload image</span></label><button data-sample>Use sample image</button><span class="status" data-image-info></span></div>
<div class="tool-group"><strong>Colors</strong><div class="row" data-colors role="group" aria-label="Color tools"></div>
<div class="tint-controls" data-tint-controls hidden><label>Tint color <input type="color" value="#ff9933" aria-label="Tint color"></label><span data-rgb></span></div></div>
<div class="tool-group" data-geometry><strong>Transformations</strong><div class="row" data-tools role="group" aria-label="Transformation tools"></div><div class="sliders" data-sliders></div></div>
<div class="row"><button data-reset>Reset tools</button><button data-save>Save canvas PNG</button></div>
<canvas width="1800" height="960" role="img" aria-label="Original image and Canva124 result on coordinate grids"></canvas>
<div class="matrices"><div>Color matrix <span data-color-matrix></span></div><div data-geometry>Position matrix <span data-position-matrix></span></div></div>
<p class="status" aria-live="polite" data-status></p>
<p class="hint" style="margin-bottom:0" data-geometry>Positions are measured from the image's center, with positive y upward. The dashed outline marks the original image.</p>
<script>
(() => {
 const root=document.getElementById('__ID__'), data=__DATA__;
 const canvas=root.querySelector('canvas'),ctx=canvas.getContext('2d'),status=root.querySelector('[data-status]');
 const source=document.createElement('canvas'),filtered=document.createElement('canvas');
 let pixels=null,original=[],geometry=0,colorIndex=0,tintSelected=false,loadVersion=0;
 const state={kind:'original',sx:1,sy:1,k:0,angle:0};
 const colorButtons=root.querySelector('[data-colors]'),toolButtons=root.querySelector('[data-tools]'),picker=root.querySelector('input[type="color"]');
 root.querySelectorAll('[data-geometry]').forEach(el=>el.hidden=data.stage<4);
 function matrixHTML(M){return '<span class="matrix" style="grid-template-columns:repeat('+M[0].length+',auto)">'+M.flat().map(x=>'<span>'+(Math.abs(x)<.0005?'0':Number(x.toFixed(3)))+'</span>').join('')+'</span>';}
 function rgb(){return [1,3,5].map(i=>parseInt(picker.value.slice(i,i+2),16));}
 function colorMap(){
  if(!tintSelected)return data.colors[colorIndex];
  const c=rgb(),combine=key=>[0,1,2].map(i=>[0,1,2].map(j=>c.reduce((sum,value,k)=>sum+value/255*data.tint[k][key][i][j],0)));
  return {label:'Tint: RGB ('+c.join(', ')+')',matrix:combine('matrix'),operator:combine('operator')};
 }
 function applyColor(){
  if(!pixels)return;
  const A=colorMap().operator,out=new Uint8ClampedArray(pixels.data.length);
  for(let i=0;i<out.length;i+=4){const r=pixels.data[i],g=pixels.data[i+1],b=pixels.data[i+2];for(let j=0;j<3;j++)out[i+j]=Math.round(A[j][0]*r+A[j][1]*g+A[j][2]*b);out[i+3]=255;}
  filtered.width=source.width;filtered.height=source.height;filtered.getContext('2d').putImageData(new ImageData(out,source.width,source.height),0,0);draw();
 }
 function loadImage(url,label){
  const version=++loadVersion,img=new Image();
  img.onload=()=>{if(version!==loadVersion)return;const ratio=Math.min(1,960/Math.max(img.naturalWidth,img.naturalHeight));source.width=Math.max(1,Math.round(img.naturalWidth*ratio));source.height=Math.max(1,Math.round(img.naturalHeight*ratio));
   const c=source.getContext('2d');c.fillStyle='#fff';c.fillRect(0,0,source.width,source.height);c.drawImage(img,0,0,source.width,source.height);pixels=c.getImageData(0,0,source.width,source.height);
   const longest=Math.max(source.width,source.height),w=source.width/longest,h=source.height/longest;original=[[-w,h],[w,h],[-w,-h],[w,-h]];
   root.querySelector('[data-image-info]').textContent=label+' · '+source.width+' × '+source.height;status.textContent='';applyColor();};
  img.onerror=()=>{if(version===loadVersion)status.textContent='That image could not be opened. Choose a PNG, JPEG, GIF, or WebP image.';};img.src=url;
 }
 root.querySelector('input[type="file"]').addEventListener('change',event=>{const file=event.target.files[0];if(!file)return;const reader=new FileReader();reader.onload=()=>loadImage(reader.result,file.name);reader.onerror=()=>status.textContent='The image could not be read. Please choose it again.';reader.readAsDataURL(file);});
 root.querySelector('[data-sample]').addEventListener('click',()=>{root.querySelector('input[type="file"]').value='';loadImage(data.sample,'Sample image');});
 function draw(){
  if(!pixels)return;
  const op=data.ops[geometry],A=op.operator,corners=original.map(([x,y])=>[A[0][0]*x+A[0][1]*y,A[1][0]*x+A[1][1]*y]);
  const maxX=Math.max(...original.map(p=>Math.abs(p[0])),...corners.map(p=>Math.abs(p[0])),.01),maxY=Math.max(...original.map(p=>Math.abs(p[1])),...corners.map(p=>Math.abs(p[1])),.01);
  const scale=.93*Math.min(195/maxX,178/maxY),ex=200/scale,ey=180/scale;
  ctx.setTransform(2,0,0,2,0,0);ctx.clearRect(0,0,900,480);ctx.fillStyle='#fff';ctx.fillRect(0,0,900,480);
  function screen(p,panel){return [(panel?675:225)+scale*p[0],277-scale*p[1]];}
  function segment(a,b,panel,color,width=1,dash=[]){const p=screen(a,panel),q=screen(b,panel);ctx.beginPath();ctx.setLineDash(dash);ctx.strokeStyle=color;ctx.lineWidth=width;ctx.moveTo(...p);ctx.lineTo(...q);ctx.stroke();ctx.setLineDash([]);}
  function outline(points,panel,color,dash=[]){ctx.beginPath();ctx.strokeStyle=color;ctx.lineWidth=1.1;ctx.setLineDash(dash);[0,1,3,2,0].forEach((i,k)=>k?ctx.lineTo(...screen(points[i],panel)):ctx.moveTo(...screen(points[i],panel)));ctx.stroke();ctx.setLineDash([]);}
  function picture(image,points,panel){const [a,b,c]=points.map(p=>screen(p,panel));ctx.save();ctx.transform((b[0]-a[0])/image.width,(b[1]-a[1])/image.width,(c[0]-a[0])/image.height,(c[1]-a[1])/image.height,a[0],a[1]);ctx.imageSmoothingEnabled=true;ctx.drawImage(image,0,0);ctx.restore();}
  ctx.textAlign='center';ctx.fillStyle='#15253a';ctx.font='18px Georgia';ctx.fillText('Original',225,27);ctx.fillText('Your result',675,27);ctx.font='12px sans-serif';ctx.fillStyle='#536278';ctx.fillText(colorMap().label,675,49,425);if(data.stage>=4)ctx.fillText(op.label,675,69,425);
  for(let panel=0;panel<2;panel++){
   for(let t=Math.ceil(-ex);t<=ex;t++)segment([t,-ey],[t,ey],panel,'#edf1f5');for(let t=Math.ceil(-ey);t<=ey;t++)segment([-ex,t],[ex,t],panel,'#edf1f5');
   segment([-ex,0],[ex,0],panel,'#adb8c6');segment([0,-ey],[0,ey],panel,'#adb8c6');
   ctx.fillText('x',screen([ex,0],panel)[0]+8,281);ctx.fillText('y',screen([0,ey],panel)[0],screen([0,ey],panel)[1]-8);
  }
  picture(source,original,0);picture(filtered,corners,1);outline(original,0,'#15253a');outline(corners,1,'#15253a');if(op.kind!=='original')outline(original,1,'#8f9baa',[5,4]);
  root.querySelector('[data-color-matrix]').innerHTML=matrixHTML(colorMap().matrix);root.querySelector('[data-position-matrix]').innerHTML=matrixHTML(op.matrix);
 }
 function syncColors(){colorButtons.querySelectorAll('button').forEach(b=>b.setAttribute('aria-pressed',b.dataset.tint==='yes'?String(tintSelected):String(!tintSelected&&Number(b.dataset.index)===colorIndex)));root.querySelector('[data-tint-controls]').hidden=!tintSelected;root.querySelector('[data-rgb]').textContent='RGB ('+rgb().join(', ')+')';}
 data.colors.forEach((item,index)=>{const b=document.createElement('button');b.textContent=item.label;b.dataset.index=index;b.addEventListener('click',()=>{colorIndex=index;tintSelected=false;syncColors();applyColor();});colorButtons.appendChild(b);});
 if(data.tint.length){const b=document.createElement('button');b.textContent='Tint';b.dataset.tint='yes';b.addEventListener('click',()=>{tintSelected=true;syncColors();applyColor();});colorButtons.appendChild(b);}
 picker.addEventListener('input',()=>{syncColors();applyColor();});picker.addEventListener('change',()=>{syncColors();applyColor();});
 const specs={stretch:[['sx','Horizontal factor',.5,2.5,.25],['sy','Vertical factor',.5,2.5,.25]],rotate:[['angle','Angle (degrees)',-180,180,15]],shear:[['k','Shear parameter k',-2,2,.25]],original:[]};
 function updateOperation(){geometry=data.ops.findIndex(op=>op.kind===state.kind&&Object.entries(op.parameters).every(([key,value])=>Math.abs(state[key]-value)<1e-8));draw();}
 function syncTools(){toolButtons.querySelectorAll('button').forEach(b=>b.setAttribute('aria-pressed',String(b.dataset.kind===state.kind)));const sliders=root.querySelector('[data-sliders]');sliders.replaceChildren();specs[state.kind].forEach(([key,label,min,max,step])=>{const row=document.createElement('label'),name=document.createElement('span'),range=document.createElement('input'),value=document.createElement('output');name.textContent=label;range.type='range';range.min=min;range.max=max;range.step=step;range.value=state[key];range.setAttribute('aria-label',label);value.textContent=range.value;range.addEventListener('input',()=>{state[key]=Number(range.value);value.textContent=range.value;updateOperation();});row.append(name,range,value);sliders.appendChild(row);});}
 const titles={original:'Original',stretch:'Stretch',rotate:'Rotate',shear:'Shear'};
 Object.keys(titles).filter(kind=>data.ops.some(op=>op.kind===kind)).forEach(kind=>{const b=document.createElement('button');b.textContent=titles[kind];b.dataset.kind=kind;b.addEventListener('click',()=>{state.kind=kind;syncTools();updateOperation();});toolButtons.appendChild(b);});
 root.querySelector('[data-reset]').addEventListener('click',()=>{colorIndex=0;tintSelected=false;geometry=0;picker.value='#ff9933';Object.assign(state,{kind:'original',sx:1,sy:1,k:0,angle:0});syncColors();syncTools();applyColor();});
 root.querySelector('[data-save]').addEventListener('click',()=>{const a=document.createElement('a');a.download='canva124-task-'+data.stage+'.png';a.href=canvas.toDataURL('image/png');a.click();});
 syncColors();syncTools();loadImage(data.sample,'Sample image');
})();
</script></div>'''

studio_image = np.array(Image.open(io.BytesIO(base64.b64decode('...'))).convert("RGB"))
print("Setup complete. The sample image is ready.")

## Task 1: A worked example – grayscale (0 pts; required)

---

In Homework 1, an RGB pixel had three components. The grayscale filter replaced each component by their average:

$$R_{\rm new}=G_{\rm new}=B_{\rm new}=\frac{R_{\rm old}+G_{\rm old}+B_{\rm old}}{3}.$$

Each output uses the same three coefficients, so the matrix is

$$G=\begin{bmatrix}1/3&1/3&1/3\\1/3&1/3&1/3\\1/3&1/3&1/3\end{bmatrix}.$$

**We've implemented this filter for you.** `grayscale_matrix()` returns $G$, and `filter_pixel(A, u)` computes $A\vec u$. The first function chooses the matrix; the second applies it. Notice that `filter_pixel` takes the matrix as an argument, so we can reuse it for other color filters.

Read the two functions below and run the cell. `np.ones((3, 3))` creates a $3\times3$ array of ones; dividing by 3 divides every entry.

In [ ]:
def filter_pixel(A, u):
    """Apply a color matrix A to one RGB column vector u."""
    return A @ u

def grayscale_matrix():
    """Average the original red, green, and blue values in every output row."""
    return np.ones((3, 3)) / 3

In [ ]:
check_task(1)

Run the next cell to open Canva124. Click **Upload image** or use the sample, then compare **Grayscale** with **Original**.

In [ ]:
show_studio(1)

<span style="color: #D55E00;"><strong>For your PDF:</strong></span> Nothing to submit for this task, but you must run the cells and try the grayscale filter before continuing.

## Task 2: Add sepia (5 pts)

---

Homework 1's sepia filter used:

$$\begin{aligned}
R_{\rm new}&=0.393R_{\rm old}+0.769G_{\rm old}+0.189B_{\rm old},\\
G_{\rm new}&=0.349R_{\rm old}+0.686G_{\rm old}+0.168B_{\rm old},\\
B_{\rm new}&=0.272R_{\rm old}+0.534G_{\rm old}+0.131B_{\rm old}.
\end{aligned}$$

Implement `sepia_matrix()`. You do **not** need a new pixel-processing function: the same `filter_pixel` should work with this new matrix. Do not normalize the rows, round values, or clip them.

**Your turn:** fill in the nine entries below, one row for each output color. Use the grayscale example as a model. Once your checks pass, click the new **Sepia** button.

In [ ]:
def sepia_matrix():
    """Return the matrix whose rows contain the coefficients in the sepia formulas."""
    return np.array([[..., ..., ...],
                     [..., ..., ...],
                     [..., ..., ...]])

In [ ]:
check_task(2)
show_studio(2)

Click **Sepia** and compare it with **Grayscale**.

<span style="color: #D55E00;"><strong>For your PDF:</strong></span> Include a screenshot of your `sepia_matrix` function and the canvas showing your image converted to sepia.

## Task 3: Choose your own tint (5 pts)

---

Let's add a color picker to Canva124. A chosen color is stored in Python as an RGB tuple, such as `(255, 153, 51)`. Each entry is between 0 and 255. This tuple specifies three numbers; when we apply a color matrix, the pixel itself is still represented by a column vector.

Our **tint** filter scales the red, green, and blue channels separately. For a chosen color `(r, g, b)`, we want

$$R_{\rm new}=\frac{r}{255}R_{\rm old},\qquad
G_{\rm new}=\frac{g}{255}G_{\rm old},\qquad
B_{\rm new}=\frac{b}{255}B_{\rm old}.$$

For example, choosing `(255, 153, 51)` keeps all of the original red, 60% of the green, and 20% of the blue.

Implement `tint_matrix(rgb)`, returning the $3\times3$ matrix for this rule. The first line unpacks the tuple into `r`, `g`, and `b`. Fill in the three missing diagonal entries; we've put in the zeros for you. Use the same `filter_pixel` function to apply this matrix.

Once your checks pass, click **Tint** in Canva124. The color picker lets you select any RGB color, and the chosen tuple appears beside it.

In [ ]:
def tint_matrix(rgb):
    """Return the color matrix for the selected (r, g, b) tuple."""
    r, g, b = rgb
    return np.array([[..., 0, 0],
                     [0, ..., 0],
                     [0, 0, ...]])

In [ ]:
check_task(3)
show_studio(3)

Try a tint on your own picture or the sample.

**Explore the extremes:** Try white `(255, 255, 255)` and black `(0, 0, 0)` in the color picker and see what happens.

**Remove one channel:** Find a color that removes all blue from the image while leaving the red and green values unchanged.

<span style="color: #D55E00;"><strong>For your PDF:</strong></span> Include a screenshot of your `tint_matrix` function and two canvas screenshots: one with a tint of your choice, and one with blue removed while red and green are unchanged. Keep the selected RGB values visible in both screenshots.

## Task 4: A worked stretch, then your own shear (5 pts)

---

A color filter changes the three entries of an RGB vector. A geometric transformation changes the two entries of a position vector $\begin{bmatrix}x\\y\end{bmatrix}$.

Here, we'll use the mathematical coordinate system, where $x$ increases rightwards and $y$ increases upwards, and the origin is at the center of the image. This is different than the standard coordinate system for images, where the origin $(0, 0)$ is the top left pixel, and the $y$ values increase downward.

**We've implemented stretching for you.** To multiply horizontal coordinates by $s_x$ and vertical coordinates by $s_y$, use

$$D=\begin{bmatrix}s_x&0\\0&s_y\end{bmatrix},\qquad D\begin{bmatrix}x\\y\end{bmatrix}=\begin{bmatrix}s_xx\\s_yy\end{bmatrix}.$$

The two sliders supply the arguments `sx` and `sy` to `scale_matrix`. A factor of 1 leaves that coordinate unchanged; a factor between 0 and 1 compresses it.

The columns of a $2\times n$ matrix $X$ hold the image's position vectors. Our `transform_points(A, X)` function computes $AX$ to transform all of those columns at once. We'll reuse this function for every geometric tool.

Run the next two cells. Click **Stretch** and move its two sliders independently. Try horizontal factor 2 and vertical factor 1.

In [ ]:
def transform_points(A, X):
    """Apply A to every position vector stored as a column of X."""
    return A @ X

def scale_matrix(sx, sy):
    """Scale x by sx and y by sy. The sliders supply these two arguments."""
    return np.array([[sx, 0],
                     [0, sy]])

In [ ]:
show_studio(4, tools=["stretch"])

### Your turn: add a shear

A horizontal shear with parameter $k$ sends

$$\begin{bmatrix}x\\y\end{bmatrix}\longmapsto\begin{bmatrix}x+ky\\y\end{bmatrix}.$$

Use this rule to fill in the four entries of `shear_matrix(k)`. Think about what happens to the two standard basis vectors; their outputs become the columns of your matrix. The slider supplies `k`, so your function must work for any value of that argument. Do not put one particular slider value into your matrix.

In [ ]:
def shear_matrix(k):
    """Return the matrix for x_new = x + k*y, y_new = y."""
    return np.array([[..., ...],
                     [..., ...]])

In [ ]:
check_task(4)
show_studio(4)

Click **Shear** to reveal the $k$ slider. Try positive and negative values of $k$. You can still change the color filter while keeping the geometric transformation fixed.

<span style="color: #D55E00;"><strong>For your PDF:</strong></span> Include a screenshot of your `shear_matrix` function and two canvas screenshots: one with a nonzero shear, and one with different horizontal and vertical stretch factors. Keep the slider values visible.

## Task 5: Add rotation (5 pts)

---

Implement `rotation_matrix(theta)` using the rotation matrix from Lab 6:

$$Q=\begin{bmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{bmatrix}.$$

The argument `theta` is measured in **radians**. Fill in the four matrix entries. Positive angles rotate counterclockwise, and negative angles rotate clockwise.

The **Rotate** button reveals an angle slider labeled in degrees. The supplied interface converts the slider's value to radians before calling your function. **Do not convert `theta` again inside your function.**

**Hint:** To take the cosine of `theta`, use `np.cos(theta)`. For sine, use `np.sin(theta)`.

As with stretching and shearing, the app passes your matrix to `transform_points`. You only need to construct the new matrix.

In [ ]:
def rotation_matrix(theta):
    """Return a counterclockwise rotation matrix; theta is in radians."""
    return np.array([[..., ...],
                     [..., ...]])

In [ ]:
check_task(5)
show_studio(5)

Click **Rotate**, then move the angle slider. Try a positive angle and a negative angle.

<span style="color: #D55E00;"><strong>For your PDF:</strong></span> Include a screenshot of your `rotation_matrix` function and the canvas showing your image rotated by a nonzero angle. Keep the angle slider and its value visible.

## Finish line

---

Submit the following under **Homework 5, Problem 11** in your PDF on Pensive. Keep your functions, images, and slider or color settings readable.

| Task | Points | Include |
|---|---:|---|
| 1. Grayscale | 0 | Required practice; nothing to submit |
| 2. Sepia | 5 | Function screenshot and sepia canvas screenshot |
| 3. Tint | 5 | Function screenshot and two canvas screenshots: a chosen tint and blue removed |
| 4. Stretching and shearing | 5 | Shear function screenshot and two canvas screenshots: a nonzero shear and unequal stretch factors |
| 5. Rotation | 5 | Function screenshot and canvas screenshot with a nonzero rotation |

**Total: 0 + 5 + 5 + 5 + 5 = 20 points.** You do not need to submit the notebook separately.